# 01 — Data Pipeline

Evidence that the data pipeline actually runs end to end: the sealed holdout is verified, the cleaned/feature dataset row counts are shown, and the positive rate is broken out by month. Everything here calls into `app.*` — no logic is reimplemented in this notebook.

In [7]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from app import config
from app.data import seal

print(f"DATA_DIR: {config.DATA_DIR}")
print(f"CLEAN_PARQUET: {config.CLEAN_PARQUET}")
print(f"FEATURES_PARQUET: {config.FEATURES_PARQUET}")
print(f"HOLDOUT_MONTHS: {config.HOLDOUT_MONTHS}")

DATA_DIR: D:\AI Transformation Bootcamp Project\ai-transformation-bootcamp\capstone-1-signalcraft\data
CLEAN_PARQUET: D:\AI Transformation Bootcamp Project\ai-transformation-bootcamp\capstone-1-signalcraft\data\processed\flights_clean.parquet
FEATURES_PARQUET: D:\AI Transformation Bootcamp Project\ai-transformation-bootcamp\projects\capstone_1\data\processed\flights_features.parquet
HOLDOUT_MONTHS: ('2026-05', '2026-06')


## Verify the sealed holdout

This checks the holdout's row keys still hash to the value recorded when it was sealed. It does **not** open the holdout — `seal.verify()` only reads `row_key` and `period`, never the label.

In [8]:
verified = seal.verify()
print(f"holdout hash verified: {verified}")

holdout hash verified: True


## Development frame: row count and positive rate by month

The development set is every cleaned row **except** the two sealed holdout months. The positive rate moving month to month is why every result in this project is reported as a lift over that month's own baseline, never a bare score.

In [9]:
frame = seal.development_frame(columns=["period", "label"])
print(f"development rows: {len(frame):,}")
print(f"overall positive rate: {frame.label.mean():.4f}")
frame.groupby("period").label.agg(["count", "mean"]).rename(columns={"mean": "positive_rate"})

development rows: 5,696,742
overall positive rate: 0.2218


,count,positive_rate
period,,
2025-07,612811,0.288853
2025-08,593733,0.225556
2025-09,558328,0.166316
2025-10,601570,0.203220
2025-11,555296,0.205602
2025-12,571924,0.267716
2026-01,517222,0.207793
2026-02,502396,0.197555
2026-03,592256,0.243190


## The feature dataset

`flights_features.parquet` is the development frame after `app.features.build.build()` — 24 features, no leakage columns, no nulls.

In [10]:
import pandas as pd

from app.features.build import FEATURE_COLUMNS

features = pd.read_parquet(config.FEATURES_PARQUET, columns=FEATURE_COLUMNS)
print(f"feature rows: {len(features):,}")
print(f"feature columns: {len(features.columns)}")
print(f"any nulls: {features.isna().any().any()}")

feature rows: 5,696,742
feature columns: 24
any nulls: False


## Holdout: sealed, not opened

This notebook does not call `seal.holdout_frame(unseal=True)`. The holdout opens exactly once, deliberately — not as a side effect of generating evidence.

In [ ]:
import json

seal_record = json.loads(config.SEAL_PATH.read_text(encoding="utf-8"))
print(f"opened: {seal_record['opened']}")
print(f"holdout rows sealed: {seal_record['holdout_rows']:,}")

opened: False
holdout rows sealed: 1,199,541


: 